# PW04 · Attack Merge And Metrics

用途：在 Colab 会话中执行 PW04 attack merge、formal overlay materialize 与 attack metrics 导出。PW04 只消费 PW02 finalize 与 thresholds，以及全部已完成的 PW03 attack shards。

边界：这是纯 merge / materialize / aggregate 阶段，不重新执行 embed、detect、calibrate，也不重新生成 clean source 或 attacked image。

## 运行参数与基础路径说明

- FAMILY_ID：切换到你要处理的 paper workflow family 时修改。
- PW04_MODE：显式选择 `prepare`、`quality_shard` 或 `finalize`。
- QUALITY_SHARD_INDEX：仅当 PW04_MODE=`quality_shard` 时生效，表示当前 worker 负责的 shard index。
- FORCE_RERUN：prepare 模式下清空既有 PW04 输出；quality_shard / finalize 模式只用于覆盖本阶段已存在工件。
- ENABLE_TAIL_ESTIMATION：在 prepare 阶段冻结 tail estimation 意图，finalize 会按 prepare manifest 复用该设置。
- REPO_BRANCH：如果你明确要切换 notebook 使用的仓库分支，再修改。
- DRIVE_PROJECT_ROOT、REPO_ROOT：仅在你的运行目录布局与默认 Colab 布局不一致时修改。

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

NOTEBOOK_NAME = "PW04_Attack_Merge_And_Metrics"
REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "organize"
DRIVE_MOUNT_ROOT = Path("/content/drive")
LOCAL_RUNTIME_ENABLED = True
LOCAL_PROJECT_ROOT = Path("/content/CEG_WM_PaperWorkflow")
PERSISTENT_DRIVE_PROJECT_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_PaperWorkflow"
DRIVE_BUNDLE_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_PaperWorkflow_Bundles"
if LOCAL_RUNTIME_ENABLED:
    DRIVE_PROJECT_ROOT = LOCAL_PROJECT_ROOT
else:
    DRIVE_PROJECT_ROOT = PERSISTENT_DRIVE_PROJECT_ROOT
REPO_ROOT = Path("/content/ceg_wm_workspace")
SCRIPT_PATH = REPO_ROOT / "paper_workflow" / "scripts" / "PW04_Attack_Merge_And_Metrics.py"
LOCAL_HF_HOME = REPO_ROOT / "huggingface_cache"
LOCAL_HF_HUB_CACHE = LOCAL_HF_HOME / "hub"
LOCAL_TRANSFORMERS_CACHE = LOCAL_HF_HOME / "transformers"

FAMILY_ID = "paper_eval_family_pilot_v1"
PW04_MODE = "quality_shard"      # 显式选择 `prepare`、`quality_shard` 或 `finalize`
QUALITY_SHARD_INDEX = 0
FORCE_RERUN = False
ENABLE_TAIL_ESTIMATION = False

def run_checked(command, cwd=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, check=True)

def print_json(title, payload):
    print(f"\n[{title}]")
    print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))

## 仓库初始化与前置检查

In [ ]:
try:
    from google.colab import drive  # type: ignore
    drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=True)
    DRIVE_MOUNT_STATUS = "mounted"
except Exception as exc:
    DRIVE_MOUNT_STATUS = f"skipped: {type(exc).__name__}: {exc}"

DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_HF_HOME.mkdir(parents=True, exist_ok=True)
LOCAL_HF_HUB_CACHE.mkdir(parents=True, exist_ok=True)
LOCAL_TRANSFORMERS_CACHE.mkdir(parents=True, exist_ok=True)

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    origin_result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"],
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    origin_url = origin_result.stdout.strip()
    if origin_url.rstrip("/") != REPO_URL.rstrip("/"):
        shutil.rmtree(REPO_ROOT)
        run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
    else:
        run_checked(["git", "fetch", "origin"], cwd=REPO_ROOT)
        run_checked(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT)
        run_checked(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_ROOT)
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from paper_workflow.scripts.pw_local_runtime import prepare_local_runtime_for_stage
from scripts.notebook_runtime_common import resolve_notebook_model_cache_layout

if LOCAL_RUNTIME_ENABLED:
    prepare_local_runtime_for_stage(
        stage_name="PW04_Attack_Merge_And_Metrics_quality_shard",
        family_id=FAMILY_ID,
        local_project_root=LOCAL_PROJECT_ROOT,
        drive_bundle_root=DRIVE_BUNDLE_ROOT,
        shard_index=QUALITY_SHARD_INDEX,
        clean_before_run=True,
        include_optional_control_negative=False,
    )

os.environ["HF_HOME"] = str(LOCAL_HF_HOME)
os.environ["HF_HUB_CACHE"] = str(LOCAL_HF_HUB_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(LOCAL_HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(LOCAL_TRANSFORMERS_CACHE)

run_checked([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], cwd=REPO_ROOT)
if (REPO_ROOT / "pyproject.toml").exists():
    run_checked([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], cwd=REPO_ROOT)
else:
    run_checked([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], cwd=REPO_ROOT)

from scripts.notebook_runtime_common import resolve_notebook_model_cache_layout

MODEL_CACHE_LAYOUT = resolve_notebook_model_cache_layout(DRIVE_MOUNT_ROOT, REPO_ROOT, create_directories=True)
LOCAL_HF_HOME = MODEL_CACHE_LAYOUT["local_hf_home"]
LOCAL_HF_HUB_CACHE = MODEL_CACHE_LAYOUT["local_hf_hub_cache"]
LOCAL_TRANSFORMERS_CACHE = MODEL_CACHE_LAYOUT["local_transformers_cache"]
os.environ["HF_HOME"] = str(LOCAL_HF_HOME)
os.environ["HF_HUB_CACHE"] = str(LOCAL_HF_HUB_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(LOCAL_HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(LOCAL_TRANSFORMERS_CACHE)

print_json(
    "repo_bootstrap",
    {
        "drive_mount_status": DRIVE_MOUNT_STATUS,
        "repo_url": REPO_URL,
        "repo_branch": REPO_BRANCH,
        "repo_root": str(REPO_ROOT),
        "script_path": str(SCRIPT_PATH),
        "local_hf_home": str(LOCAL_HF_HOME),
        "local_hf_hub_cache": str(LOCAL_HF_HUB_CACHE),
        "local_transformers_cache": str(LOCAL_TRANSFORMERS_CACHE),
    },
)

## Colab 质量模型依赖补齐与即时检查

说明：PW04 notebook 在 Colab 中默认先执行 editable install，而仓库的 pyproject 只声明最小核心依赖，因此这里显式补装 LPIPS 与 open_clip_torch。

目标：在继续 PW04 precheck 之前，先确认当前 notebook 内核已经能够 import 并实例化 LPIPS / CLIP 质量模型加载入口。

In [ ]:
!pip install -q lpips open_clip_torch

from pathlib import Path
import importlib
import json
import os
import sys

quality_repo_root = Path(globals().get("REPO_ROOT", Path.cwd())).expanduser().resolve()
if not (quality_repo_root / "paper_workflow").exists():
    for candidate_root in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate_root / "paper_workflow").exists() and (candidate_root / "paper_workflow" / "scripts").exists():
            quality_repo_root = candidate_root
            break

if str(quality_repo_root) not in sys.path:
    sys.path.insert(0, str(quality_repo_root))

from paper_workflow.scripts.pw_quality_metrics import (
    _get_clip_model_components,
    _get_lpips_model,
    CLIP_MODEL_NAME,
    QUALITY_CLIP_BATCH_SIZE_ENV,
    QUALITY_LPIPS_BATCH_SIZE_ENV,
    QUALITY_TORCH_DEVICE_ENV,
    DEFAULT_QUALITY_TORCH_DEVICE,
    DEFAULT_QUALITY_BATCH_SIZE,
    _resolve_quality_runtime_options,
    _LPIPS_MODEL_ERRORS,
    _CLIP_MODEL_ERRORS,
    _LPIPS_MODELS,
    _CLIP_MODELS,
    _CLIP_PREPROCESSES,
    _CLIP_TOKENIZERS,
 )

for cache_mapping in [
    _LPIPS_MODELS,
    _CLIP_MODELS,
    _CLIP_PREPROCESSES,
    _CLIP_TOKENIZERS,
    _LPIPS_MODEL_ERRORS,
    _CLIP_MODEL_ERRORS,
]:
    cache_mapping.clear()

quality_runtime = _resolve_quality_runtime_options(
    torch_device=os.environ.get(QUALITY_TORCH_DEVICE_ENV, DEFAULT_QUALITY_TORCH_DEVICE),
    lpips_batch_size=int(os.environ.get(QUALITY_LPIPS_BATCH_SIZE_ENV, DEFAULT_QUALITY_BATCH_SIZE)),
    clip_batch_size=int(os.environ.get(QUALITY_CLIP_BATCH_SIZE_ENV, DEFAULT_QUALITY_BATCH_SIZE)),
)

import_results = {}
for module_name in ["lpips", "open_clip", "torch"]:
    try:
        module_obj = importlib.import_module(module_name)
        import_results[module_name] = {
            "ok": True,
            "version": getattr(module_obj, "__version__", "unknown"),
        }
    except Exception as exc:
        import_results[module_name] = {
            "ok": False,
            "error": f"{type(exc).__name__}: {exc}",
        }

lpips_model, lpips_error = _get_lpips_model(torch_device=quality_runtime["torch_device"])
clip_model, clip_preprocess, clip_tokenizer, clip_error = _get_clip_model_components(
    torch_device=quality_runtime["torch_device"]
)

QUALITY_DEPENDENCY_CHECK = {
    "repo_root": str(quality_repo_root),
    "quality_runtime": quality_runtime,
    "imports": import_results,
    "lpips_loaded": lpips_model is not None,
    "lpips_error": lpips_error,
    "clip_loaded": clip_model is not None and clip_preprocess is not None and clip_tokenizer is not None,
    "clip_error": clip_error,
    "clip_model_name": CLIP_MODEL_NAME,
}

print("\n[quality_dependency_check]")
print(json.dumps(QUALITY_DEPENDENCY_CHECK, ensure_ascii=False, indent=2, sort_keys=True))

if not QUALITY_DEPENDENCY_CHECK["lpips_loaded"]:
    raise RuntimeError(f"LPIPS bootstrap failed: {QUALITY_DEPENDENCY_CHECK['lpips_error']}")
if not QUALITY_DEPENDENCY_CHECK["clip_loaded"]:
    raise RuntimeError(f"CLIP bootstrap failed: {QUALITY_DEPENDENCY_CHECK['clip_error']}")

In [ ]:
FAMILY_ROOT = DRIVE_PROJECT_ROOT / "paper_workflow" / "families" / FAMILY_ID
PW02_SUMMARY_PATH = FAMILY_ROOT / "runtime_state" / "pw02_summary.json"
FINALIZE_MANIFEST_PATH = FAMILY_ROOT / "exports" / "pw02" / "paper_source_finalize_manifest.json"
CONTENT_THRESHOLD_EXPORT_PATH = FAMILY_ROOT / "exports" / "pw02" / "thresholds" / "content" / "thresholds.json"
ATTESTATION_THRESHOLD_EXPORT_PATH = FAMILY_ROOT / "exports" / "pw02" / "thresholds" / "attestation" / "thresholds.json"
ATTACK_SHARD_PLAN_PATH = FAMILY_ROOT / "manifests" / "attack_shard_plan.json"
ATTACK_EVENT_GRID_PATH = FAMILY_ROOT / "manifests" / "attack_event_grid.jsonl"
PREPARE_MANIFEST_PATH = FAMILY_ROOT / "exports" / "pw04" / "manifests" / "pw04_prepare_manifest.json"
QUALITY_ROOT = FAMILY_ROOT / "exports" / "pw04" / "quality"
QUALITY_PAIR_PLAN_PATH = QUALITY_ROOT / "quality_pair_plan.json"
SELECTED_QUALITY_SHARD_PATH = QUALITY_ROOT / "shards" / f"quality_shard_{QUALITY_SHARD_INDEX:04d}.json"
QUALITY_FINALIZE_MANIFEST_PATH = QUALITY_ROOT / "quality_finalize_manifest.json"
PW04_SUMMARY_PATH = FAMILY_ROOT / "runtime_state" / "pw04_summary.json"
PROJECT_ROOT_PRECHECK_LABEL = "项目运行根目录存在" if LOCAL_RUNTIME_ENABLED else "Drive 项目根目录存在"

PRECHECK_RESULTS = []

def record_precheck(name, ok, detail):
    PRECHECK_RESULTS.append({
        "name": str(name),
        "ok": bool(ok),
        "detail": str(detail),
    })

record_precheck("PW04_MODE 合法", PW04_MODE in {"prepare", "quality_shard", "finalize"}, PW04_MODE)
record_precheck(
    "QUALITY_SHARD_INDEX 合法（worker 模式）",
    PW04_MODE != "quality_shard" or (isinstance(QUALITY_SHARD_INDEX, int) and QUALITY_SHARD_INDEX >= 0),
    str(QUALITY_SHARD_INDEX),
)

ATTACK_SHARD_PLAN = {}
EXPECTED_ATTACK_EVENT_COUNT = None
DISCOVERED_ATTACK_EVENT_COUNT = 0
PLANNED_SHARD_RESULTS = []
PREPARE_MANIFEST = {}
EXPECTED_QUALITY_SHARD_PATHS = []
if ATTACK_SHARD_PLAN_PATH.exists():
    ATTACK_SHARD_PLAN = json.loads(ATTACK_SHARD_PLAN_PATH.read_text(encoding="utf-8"))
    EXPECTED_ATTACK_EVENT_COUNT = ATTACK_SHARD_PLAN.get("attack_event_count")
    for shard_row in ATTACK_SHARD_PLAN.get("shards", []):
        shard_index = int(shard_row["attack_shard_index"])
        shard_manifest_path = FAMILY_ROOT / "attack_shards" / f"shard_{shard_index:04d}" / "shard_manifest.json"
        shard_exists = shard_manifest_path.exists()
        shard_status = "<absent>"
        shard_event_count = 0
        if shard_exists:
            shard_manifest = json.loads(shard_manifest_path.read_text(encoding="utf-8"))
            shard_status = str(shard_manifest.get("status", "<absent>"))
            if isinstance(shard_manifest.get("event_count"), int):
                shard_event_count = int(shard_manifest["event_count"])
            else:
                shard_event_count = len(shard_manifest.get("events", []))
            if shard_status == "completed":
                DISCOVERED_ATTACK_EVENT_COUNT += shard_event_count
        PLANNED_SHARD_RESULTS.append({
            "attack_shard_index": shard_index,
            "path": str(shard_manifest_path),
            "exists": shard_exists,
            "status": shard_status,
            "event_count": shard_event_count,
        })

if PREPARE_MANIFEST_PATH.exists():
    PREPARE_MANIFEST = json.loads(PREPARE_MANIFEST_PATH.read_text(encoding="utf-8"))
    EXPECTED_QUALITY_SHARD_PATHS = [
        Path(str(path_value))
        for path_value in PREPARE_MANIFEST.get("expected_quality_shard_paths", [])
    ]

if PW04_MODE == "prepare":
    record_precheck(PROJECT_ROOT_PRECHECK_LABEL, DRIVE_PROJECT_ROOT.exists(), str(DRIVE_PROJECT_ROOT))
    record_precheck("仓库目录存在", REPO_ROOT.exists(), str(REPO_ROOT))
    record_precheck("PW04 脚本存在", SCRIPT_PATH.exists(), str(SCRIPT_PATH))
    record_precheck("PW02_SUMMARY_PATH 存在", PW02_SUMMARY_PATH.exists(), str(PW02_SUMMARY_PATH))
    record_precheck("FINALIZE_MANIFEST_PATH 存在", FINALIZE_MANIFEST_PATH.exists(), str(FINALIZE_MANIFEST_PATH))
    record_precheck("content thresholds export 存在", CONTENT_THRESHOLD_EXPORT_PATH.exists(), str(CONTENT_THRESHOLD_EXPORT_PATH))
    record_precheck("attestation thresholds export 存在", ATTESTATION_THRESHOLD_EXPORT_PATH.exists(), str(ATTESTATION_THRESHOLD_EXPORT_PATH))
    record_precheck("ATTACK_SHARD_PLAN_PATH 存在", ATTACK_SHARD_PLAN_PATH.exists(), str(ATTACK_SHARD_PLAN_PATH))
    record_precheck("ATTACK_EVENT_GRID_PATH 存在", ATTACK_EVENT_GRID_PATH.exists(), str(ATTACK_EVENT_GRID_PATH))
    record_precheck(
        "所有计划内 PW03 shard manifest 存在且 completed",
        all(item["exists"] and item["status"] == "completed" for item in PLANNED_SHARD_RESULTS),
        json.dumps(PLANNED_SHARD_RESULTS, ensure_ascii=False, indent=2),
    )
    record_precheck(
        "expected_attack_event_count == discovered_attack_event_count",
        isinstance(EXPECTED_ATTACK_EVENT_COUNT, int) and EXPECTED_ATTACK_EVENT_COUNT == DISCOVERED_ATTACK_EVENT_COUNT,
        f"expected_attack_event_count={EXPECTED_ATTACK_EVENT_COUNT}, discovered_attack_event_count={DISCOVERED_ATTACK_EVENT_COUNT}",
    )
elif PW04_MODE == "quality_shard":
    record_precheck("PREPARE_MANIFEST_PATH 存在", PREPARE_MANIFEST_PATH.exists(), str(PREPARE_MANIFEST_PATH))
    record_precheck("QUALITY_PAIR_PLAN_PATH 存在", QUALITY_PAIR_PLAN_PATH.exists(), str(QUALITY_PAIR_PLAN_PATH))
    record_precheck("PW04 summary 尚未关闭（worker 模式）", not PW04_SUMMARY_PATH.exists(), str(PW04_SUMMARY_PATH))
    record_precheck(
        "prepare manifest 已声明当前 quality shard",
        any(Path(str(path_obj)).name == SELECTED_QUALITY_SHARD_PATH.name for path_obj in EXPECTED_QUALITY_SHARD_PATHS),
        json.dumps([str(path_obj) for path_obj in EXPECTED_QUALITY_SHARD_PATHS], ensure_ascii=False, indent=2),
    )
else:
    record_precheck("PREPARE_MANIFEST_PATH 存在", PREPARE_MANIFEST_PATH.exists(), str(PREPARE_MANIFEST_PATH))
    record_precheck("QUALITY_PAIR_PLAN_PATH 存在", QUALITY_PAIR_PLAN_PATH.exists(), str(QUALITY_PAIR_PLAN_PATH))
    record_precheck(
        "prepare manifest 已声明 quality shards",
        bool(EXPECTED_QUALITY_SHARD_PATHS),
        json.dumps([str(path_obj) for path_obj in EXPECTED_QUALITY_SHARD_PATHS], ensure_ascii=False, indent=2),
    )
    record_precheck(
        "全部计划内 quality shard 已完成",
        bool(EXPECTED_QUALITY_SHARD_PATHS) and all(path_obj.exists() for path_obj in EXPECTED_QUALITY_SHARD_PATHS),
        json.dumps([str(path_obj) for path_obj in EXPECTED_QUALITY_SHARD_PATHS], ensure_ascii=False, indent=2),
    )
    record_precheck(
        "PW04_SUMMARY_PATH 尚未存在或允许覆盖",
        FORCE_RERUN or not PW04_SUMMARY_PATH.exists(),
        str(PW04_SUMMARY_PATH),
    )

PW04_PRECHECK_CONTEXT = {
    "pw04_mode": PW04_MODE,
    "quality_shard_index": QUALITY_SHARD_INDEX,
    "prepare_manifest_path": str(PREPARE_MANIFEST_PATH),
    "quality_pair_plan_path": str(QUALITY_PAIR_PLAN_PATH),
    "selected_quality_shard_path": str(SELECTED_QUALITY_SHARD_PATH),
    "quality_finalize_manifest_path": str(QUALITY_FINALIZE_MANIFEST_PATH),
    "pw04_summary_path": str(PW04_SUMMARY_PATH),
    "expected_quality_shard_paths": [str(path_obj) for path_obj in EXPECTED_QUALITY_SHARD_PATHS],
}

print_json("pw04_precheck", {"items": PRECHECK_RESULTS, "context": PW04_PRECHECK_CONTEXT})
hard_fail = [item for item in PRECHECK_RESULTS if not item["ok"]]
if hard_fail:
    raise RuntimeError(f"PW04 precheck failed: {hard_fail}")

## Quality Runtime 配置说明

- QUALITY_DEVICE_OVERRIDE：可选 auto、cuda、cpu。默认建议保持 auto。
- 如果你非常明确要覆盖 batch，可以在运行本 cell 前预先设置环境变量 PW_QUALITY_LPIPS_BATCH_SIZE 与 PW_QUALITY_CLIP_BATCH_SIZE。
- 一般情况下不需要手改默认 batch 选择逻辑；只有在你已经确认显存非常紧张或非常充足时，才建议显式覆盖。

In [ ]:
import json
import os

QUALITY_DEVICE_OVERRIDE = "auto"
VALID_QUALITY_DEVICE_REQUESTS = {"auto", "cuda", "cpu"}

if "print_json" not in globals():
    def print_json(title, payload):
        print(f"\n[{title}]")
        print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))

def normalize_quality_device_request(raw_value, label, *, default_value="auto"):
    if raw_value is None:
        return None, None
    if not isinstance(raw_value, str):
        return default_value, f"{label} must be one of auto/cuda/cpu; fallback to {default_value}"
    normalized_value = raw_value.strip().lower()
    if not normalized_value:
        return default_value, f"{label} is empty; fallback to {default_value}"
    if normalized_value not in VALID_QUALITY_DEVICE_REQUESTS:
        return default_value, f"{label}={raw_value!r} invalid; fallback to {default_value}"
    return normalized_value, None

def normalize_positive_batch_size(raw_value, label, default_value):
    if raw_value is None:
        return default_value, None, "device_default"
    try:
        normalized_value = int(str(raw_value).strip())
    except (TypeError, ValueError):
        return default_value, f"{label}={raw_value!r} invalid; fallback to {default_value}", "device_default"
    if normalized_value <= 0:
        return default_value, f"{label}={raw_value!r} invalid; fallback to {default_value}", "device_default"
    return normalized_value, None, "environment"

quality_warnings = []

resolved_notebook_request, notebook_warning = normalize_quality_device_request(
    QUALITY_DEVICE_OVERRIDE,
    "QUALITY_DEVICE_OVERRIDE",
    default_value="auto",
)
if notebook_warning is not None:
    quality_warnings.append(notebook_warning)

resolved_env_request = None
env_requested_device_value = os.environ.get("PW_QUALITY_TORCH_DEVICE")
if env_requested_device_value is not None:
    resolved_env_request, env_warning = normalize_quality_device_request(
        env_requested_device_value,
        "PW_QUALITY_TORCH_DEVICE",
        default_value="auto",
    )
    if env_warning is not None:
        quality_warnings.append(env_warning)

if resolved_notebook_request is not None and resolved_notebook_request != "auto":
    requested_device = resolved_notebook_request
    requested_device_source = "notebook_override"
elif resolved_env_request is not None:
    requested_device = resolved_env_request
    requested_device_source = "environment"
else:
    requested_device = "auto"
    requested_device_source = "notebook_default_auto"

detected_cuda_available = False
detected_cuda_device_count = 0
detected_cuda_device_name = None
detected_cuda_total_memory_gib = None
torch_runtime_status = "not_imported"
detection_reason = ""
selection_reason = ""
fallback_reason = None

try:
    import torch
except Exception as exc:
    torch_runtime_status = f"import_failed: {type(exc).__name__}: {exc}"
    detection_reason = f"torch import failed; treat cuda as unavailable: {type(exc).__name__}: {exc}"
else:
    torch_runtime_status = "imported"
    try:
        raw_cuda_available = bool(torch.cuda.is_available())
        detected_cuda_device_count = int(torch.cuda.device_count())
        detected_cuda_available = bool(raw_cuda_available and detected_cuda_device_count > 0)
    except Exception as exc:
        detected_cuda_available = False
        detected_cuda_device_count = 0
        detection_reason = f"torch.cuda probe failed; treat cuda as unavailable: {type(exc).__name__}: {exc}"
    else:
        if detected_cuda_available:
            try:
                detected_cuda_device_name = torch.cuda.get_device_name(0)
            except Exception:
                detected_cuda_device_name = None
            try:
                detected_cuda_total_memory_gib = round(
                    float(torch.cuda.get_device_properties(0).total_memory) / float(1024 ** 3),
                    2,
                )
            except Exception:
                detected_cuda_total_memory_gib = None
            detection_reason = "torch.cuda.is_available() returned True and at least one CUDA device is available"
        elif raw_cuda_available:
            detection_reason = "torch.cuda.is_available() returned True but no CUDA devices were enumerated"
        else:
            detection_reason = "torch.cuda.is_available() returned False"

if requested_device == "cpu":
    selected_device = "cpu"
    selection_reason = "cpu explicitly requested"
elif requested_device == "cuda":
    if detected_cuda_available:
        selected_device = "cuda"
        selection_reason = "requested cuda and cuda is available"
    else:
        selected_device = "cpu"
        selection_reason = "requested cuda but unavailable"
        fallback_reason = "requested cuda but unavailable; fallback to cpu"
        quality_warnings.append(fallback_reason)
else:
    if detected_cuda_available:
        selected_device = "cuda"
        selection_reason = "auto selected cuda because cuda is available"
    else:
        selected_device = "cpu"
        selection_reason = "auto selected cpu because cuda is unavailable"

if selected_device == "cpu":
    default_lpips_batch_size = 1
    default_clip_batch_size = 1
    batch_default_reason = "cpu runtime uses single-item batch defaults"
elif detected_cuda_total_memory_gib is not None and detected_cuda_total_memory_gib < 12.0:
    default_lpips_batch_size = 4
    default_clip_batch_size = 8
    batch_default_reason = (
        f"cuda runtime with approximately {detected_cuda_total_memory_gib} GiB memory uses conservative GPU batch defaults"
    )
else:
    default_lpips_batch_size = 16
    default_clip_batch_size = 32
    batch_default_reason = "cuda runtime uses default GPU batch sizes"

lpips_batch_size, lpips_warning, lpips_batch_size_source = normalize_positive_batch_size(
    os.environ.get("PW_QUALITY_LPIPS_BATCH_SIZE"),
    "PW_QUALITY_LPIPS_BATCH_SIZE",
    default_lpips_batch_size,
)
if lpips_warning is not None:
    quality_warnings.append(lpips_warning)

clip_batch_size, clip_warning, clip_batch_size_source = normalize_positive_batch_size(
    os.environ.get("PW_QUALITY_CLIP_BATCH_SIZE"),
    "PW_QUALITY_CLIP_BATCH_SIZE",
    default_clip_batch_size,
)
if clip_warning is not None:
    quality_warnings.append(clip_warning)

os.environ["PW_QUALITY_TORCH_DEVICE"] = selected_device
os.environ["PW_QUALITY_LPIPS_BATCH_SIZE"] = str(lpips_batch_size)
os.environ["PW_QUALITY_CLIP_BATCH_SIZE"] = str(clip_batch_size)

QUALITY_RUNTIME_SUMMARY = {
    "requested_device": requested_device,
    "requested_device_source": requested_device_source,
    "detected_cuda_available": detected_cuda_available,
    "detected_cuda_device_count": detected_cuda_device_count,
    "detected_cuda_device_name": detected_cuda_device_name,
    "detected_cuda_total_memory_gib": detected_cuda_total_memory_gib,
    "selected_device": selected_device,
    "lpips_batch_size": lpips_batch_size,
    "lpips_batch_size_source": lpips_batch_size_source,
    "clip_batch_size": clip_batch_size,
    "clip_batch_size_source": clip_batch_size_source,
    "torch_runtime_status": torch_runtime_status,
    "detection_reason": detection_reason,
    "selection_reason": selection_reason,
    "fallback_reason": fallback_reason,
    "batch_default_reason": batch_default_reason,
    "warnings": quality_warnings,
}

print_json("pw04_quality_runtime", QUALITY_RUNTIME_SUMMARY)

## PW04 主执行

In [ ]:
from paper_workflow.scripts.pw_local_runtime import archive_local_runtime_for_stage
from scripts.notebook_runtime_common import build_repo_import_subprocess_env

try:
    from scripts.gpu_session_peak import build_gpu_peak_display_summary as _build_gpu_peak_display_summary
except Exception as exc:
    GPU_PEAK_DISPLAY_HELPER = None
    GPU_PEAK_DISPLAY_HELPER_IMPORT_ERROR = f"{type(exc).__name__}: {exc}"
else:
    GPU_PEAK_DISPLAY_HELPER = _build_gpu_peak_display_summary
    GPU_PEAK_DISPLAY_HELPER_IMPORT_ERROR = None

COMMAND = [
    sys.executable,
    str(SCRIPT_PATH),
    "--drive-project-root",
    str(DRIVE_PROJECT_ROOT),
    "--family-id",
    FAMILY_ID,
    "--pw04-mode",
    PW04_MODE,
]
if PW04_MODE == "quality_shard":
    COMMAND.extend(["--quality-shard-index", str(QUALITY_SHARD_INDEX)])
if FORCE_RERUN:
    COMMAND.append("--force-rerun")
if ENABLE_TAIL_ESTIMATION:
    COMMAND.append("--enable-tail-estimation")

if PW04_MODE == "prepare":
    EXPECTED_OUTPUT_LABEL = "prepare_manifest"
    EXPECTED_OUTPUT_PATH = PREPARE_MANIFEST_PATH
elif PW04_MODE == "quality_shard":
    EXPECTED_OUTPUT_LABEL = "quality_shard"
    EXPECTED_OUTPUT_PATH = SELECTED_QUALITY_SHARD_PATH
else:
    EXPECTED_OUTPUT_LABEL = "pw04_summary"
    EXPECTED_OUTPUT_PATH = PW04_SUMMARY_PATH

PW04_NOTEBOOK_QUALITY_RUNTIME = globals().get("QUALITY_RUNTIME_SUMMARY", {})
if not isinstance(PW04_NOTEBOOK_QUALITY_RUNTIME, dict):
    PW04_NOTEBOOK_QUALITY_RUNTIME = {}

PW04_SUBPROCESS_ENV = build_repo_import_subprocess_env(
    base_env=os.environ,
    repo_root=REPO_ROOT,
    )
PW04_SUBPROCESS_QUALITY_RUNTIME = {
    "requested_device": PW04_NOTEBOOK_QUALITY_RUNTIME.get("requested_device", "<absent>"),
    "detected_cuda_available": PW04_NOTEBOOK_QUALITY_RUNTIME.get("detected_cuda_available", "<absent>"),
    "selected_device": PW04_SUBPROCESS_ENV.get("PW_QUALITY_TORCH_DEVICE", "<absent>"),
    "lpips_batch_size": PW04_SUBPROCESS_ENV.get("PW_QUALITY_LPIPS_BATCH_SIZE", "<absent>"),
    "clip_batch_size": PW04_SUBPROCESS_ENV.get("PW_QUALITY_CLIP_BATCH_SIZE", "<absent>"),
    "selection_reason": PW04_NOTEBOOK_QUALITY_RUNTIME.get("selection_reason", "<absent>"),
    "fallback_reason": PW04_NOTEBOOK_QUALITY_RUNTIME.get("fallback_reason", "<absent>"),
    "warnings": PW04_NOTEBOOK_QUALITY_RUNTIME.get("warnings", []),
}
print_json("pw04_subprocess_quality_runtime", PW04_SUBPROCESS_QUALITY_RUNTIME)

GPU_PEAK_SCRIPT_PATH = REPO_ROOT / "scripts" / "gpu_session_peak.py"
if PW04_MODE == "quality_shard":
    GPU_PEAK_PATH_STEM = f"pw04_quality_shard_{QUALITY_SHARD_INDEX:04d}"
else:
    GPU_PEAK_PATH_STEM = f"pw04_{PW04_MODE}"
GPU_PEAK_SUMMARY_PATH = FAMILY_ROOT / "runtime_state" / f"{GPU_PEAK_PATH_STEM}_gpu_session_peak.json"
GPU_PEAK_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
if GPU_PEAK_SUMMARY_PATH.exists():
    GPU_PEAK_SUMMARY_PATH.unlink()

MONITORED_COMMAND = [
    sys.executable,
    str(GPU_PEAK_SCRIPT_PATH),
    "--output-json",
    str(GPU_PEAK_SUMMARY_PATH),
    "--label",
    NOTEBOOK_NAME,
    "--sample-interval-ms",
    "200",
    "--",
    *COMMAND,
]

def load_gpu_peak_summary(path_obj):
    if not path_obj.exists() or not path_obj.is_file():
        return None, f"gpu peak summary missing: {path_obj}"
    try:
        payload = json.loads(path_obj.read_text(encoding="utf-8"))
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"
    if not isinstance(payload, dict):
        return None, f"gpu peak summary must be JSON object: {path_obj}"
    return payload, None

def build_fallback_display_summary(monitor_status):
    summary_payload = {
        "status": monitor_status,
        "session_board_peak_memory_used_mib": None,
        "session_board_memory_used_mib_at_start": None,
        "session_board_memory_used_mib_at_end": None,
        "peak_gpu_name": None,
        "peak_gpu_index": None,
        "visible_gpu_count": 0,
    }
    if GPU_PEAK_DISPLAY_HELPER is not None:
        return GPU_PEAK_DISPLAY_HELPER(summary_payload)
    return {
        "status": monitor_status,
        "peak_memory_used_mib": None,
        "peak_memory_used_gib": None,
        "start_memory_used_mib": None,
        "end_memory_used_mib": None,
        "peak_gpu_name": None,
        "peak_gpu_index": None,
        "visible_gpu_count": 0,
        "recommendation": "未取得 nvidia-smi 峰值，暂时无法给出显存档位提示",
    }

def build_gpu_peak_notebook_summary(raw_summary, *, monitor_status, monitor_error, fallback_reason):
    if isinstance(raw_summary, dict):
        if GPU_PEAK_DISPLAY_HELPER is not None:
            display_summary = GPU_PEAK_DISPLAY_HELPER(raw_summary)
        else:
            display_summary = build_fallback_display_summary(str(raw_summary.get("status", monitor_status)))
        resolved_status = str(raw_summary.get("status", display_summary.get("status", monitor_status)))
        return {
            "gpu_session_peak_path": str(GPU_PEAK_SUMMARY_PATH),
            "gpu_peak_memory_mib": raw_summary.get("session_board_peak_memory_used_mib"),
            "gpu_peak_increment_mib": raw_summary.get("session_board_peak_increment_mib"),
            "peak_gpu_index": raw_summary.get("peak_gpu_index"),
            "peak_gpu_uuid": raw_summary.get("peak_gpu_uuid"),
            "peak_gpu_name": raw_summary.get("peak_gpu_name"),
            "monitor_status": resolved_status,
            "monitor_recommendation": display_summary.get("recommendation"),
            "monitor_error": monitor_error or raw_summary.get("monitor_error"),
            "visible_gpu_count": raw_summary.get("visible_gpu_count"),
            "wrapped_return_code": raw_summary.get("wrapped_return_code"),
            "monitor_fallback_reason": fallback_reason,
        }
    display_summary = build_fallback_display_summary(monitor_status)
    return {
        "gpu_session_peak_path": str(GPU_PEAK_SUMMARY_PATH),
        "gpu_peak_memory_mib": display_summary.get("peak_memory_used_mib"),
        "gpu_peak_increment_mib": None,
        "peak_gpu_index": display_summary.get("peak_gpu_index"),
        "peak_gpu_uuid": None,
        "peak_gpu_name": display_summary.get("peak_gpu_name"),
        "monitor_status": monitor_status,
        "monitor_recommendation": display_summary.get("recommendation"),
        "monitor_error": monitor_error,
        "visible_gpu_count": display_summary.get("visible_gpu_count"),
        "wrapped_return_code": None,
        "monitor_fallback_reason": fallback_reason,
    }

PW04_GPU_MONITOR_SETUP = {
    "gpu_peak_script_path": str(GPU_PEAK_SCRIPT_PATH),
    "gpu_peak_summary_path": str(GPU_PEAK_SUMMARY_PATH),
    "gpu_peak_script_exists": GPU_PEAK_SCRIPT_PATH.exists(),
    "gpu_peak_display_helper_import_error": GPU_PEAK_DISPLAY_HELPER_IMPORT_ERROR,
    "monitored_command": MONITORED_COMMAND,
    "expected_output_label": EXPECTED_OUTPUT_LABEL,
    "expected_output_path": str(EXPECTED_OUTPUT_PATH),
}
print_json("pw04_gpu_monitor_setup", PW04_GPU_MONITOR_SETUP)

PW04_MONITOR_RAW_SUMMARY = None
PW04_MONITOR_LOAD_ERROR = None
PW04_MONITOR_FALLBACK_REASON = None
PW04_MONITOR_EXECUTION_MODE = "wrapped_command"

if not GPU_PEAK_SCRIPT_PATH.exists():
    PW04_MONITOR_EXECUTION_MODE = "direct_command_script_missing"
    PW04_MONITOR_FALLBACK_REASON = f"gpu_session_peak.py missing: {GPU_PEAK_SCRIPT_PATH}"
    PW04_RESULT = subprocess.run(
        COMMAND,
        cwd=REPO_ROOT,
        env=PW04_SUBPROCESS_ENV,
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
else:
    PW04_RESULT = subprocess.run(
        MONITORED_COMMAND,
        cwd=REPO_ROOT,
        env=PW04_SUBPROCESS_ENV,
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    PW04_MONITOR_RAW_SUMMARY, PW04_MONITOR_LOAD_ERROR = load_gpu_peak_summary(GPU_PEAK_SUMMARY_PATH)
    wrapped_command_started = (
        isinstance(PW04_MONITOR_RAW_SUMMARY, dict)
        and PW04_MONITOR_RAW_SUMMARY.get("wrapped_return_code") is not None
    )
    if PW04_RESULT.returncode != 0 and not wrapped_command_started:
        PW04_MONITOR_EXECUTION_MODE = "direct_command_after_wrapper_failure"
        PW04_MONITOR_FALLBACK_REASON = (
            "gpu_session_peak wrapper failed before the wrapped PW04 command produced a return code; rerun bare PW04 command"
        )
        PW04_RESULT = subprocess.run(
            COMMAND,
            cwd=REPO_ROOT,
            env=PW04_SUBPROCESS_ENV,
            check=False,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
        )

GPU_PEAK_NOTEBOOK_SUMMARY = build_gpu_peak_notebook_summary(
    PW04_MONITOR_RAW_SUMMARY,
    monitor_status=(
        str(PW04_MONITOR_RAW_SUMMARY.get("status"))
        if isinstance(PW04_MONITOR_RAW_SUMMARY, dict) and isinstance(PW04_MONITOR_RAW_SUMMARY.get("status"), str)
        else ("skipped" if PW04_MONITOR_EXECUTION_MODE.startswith("direct_command") else "unavailable")
    ),
    monitor_error=PW04_MONITOR_LOAD_ERROR or GPU_PEAK_DISPLAY_HELPER_IMPORT_ERROR or PW04_MONITOR_FALLBACK_REASON,
    fallback_reason=PW04_MONITOR_FALLBACK_REASON,
 )
GPU_PEAK_NOTEBOOK_SUMMARY["monitor_execution_mode"] = PW04_MONITOR_EXECUTION_MODE
print_json("gpu_session_peak_summary", GPU_PEAK_NOTEBOOK_SUMMARY)

if PW04_RESULT.returncode != 0:
    raise RuntimeError(
        "PW04 failed: "
        f"return_code={PW04_RESULT.returncode} "
        f"monitor_status={GPU_PEAK_NOTEBOOK_SUMMARY.get('monitor_status')} "
        f"monitor_execution_mode={PW04_MONITOR_EXECUTION_MODE} "
        f"stdout={PW04_RESULT.stdout} stderr={PW04_RESULT.stderr}"
    )
if not EXPECTED_OUTPUT_PATH.exists():
    raise FileNotFoundError(
        f"PW04 {EXPECTED_OUTPUT_LABEL} file missing after successful execution: {EXPECTED_OUTPUT_PATH}"
    )

PW04_MODE_OUTPUT = json.loads(EXPECTED_OUTPUT_PATH.read_text(encoding="utf-8"))
if PW04_MODE == "finalize":
    PW04_SUMMARY = dict(PW04_MODE_OUTPUT)
else:
    PW04_SUMMARY = None
print_json("pw04_mode_output", PW04_MODE_OUTPUT)
if LOCAL_RUNTIME_ENABLED:
    archive_local_runtime_for_stage(
        stage_name="PW04_Attack_Merge_And_Metrics_quality_shard",
        family_id=FAMILY_ID,
        local_project_root=LOCAL_PROJECT_ROOT,
        drive_bundle_root=DRIVE_BUNDLE_ROOT,
        shard_index=QUALITY_SHARD_INDEX,
        clean_after_archive=False,
    )

## 输出读取与结果检查

In [ ]:
FORMAL_ATTACK_FINAL_DECISION_METRICS_PATH = FAMILY_ROOT / "exports" / "pw04" / "formal_attack_final_decision_metrics.json"
FORMAL_ATTACK_ATTESTATION_METRICS_PATH = FAMILY_ROOT / "exports" / "pw04" / "formal_attack_attestation_metrics.json"
DERIVED_ATTACK_UNION_METRICS_PATH = FAMILY_ROOT / "exports" / "pw04" / "derived_attack_union_metrics.json"
CLEAN_ATTACK_OVERVIEW_PATH = FAMILY_ROOT / "exports" / "pw04" / "clean_attack_overview.json"

if PW04_MODE == "prepare":
    PREPARE_MANIFEST = json.loads(PREPARE_MANIFEST_PATH.read_text(encoding="utf-8"))
    PW04_RESULT_SUMMARY = {
        "mode": PW04_MODE,
        "prepare_manifest": json.loads(PREPARE_MANIFEST_PATH.read_text(encoding="utf-8")),
        "attack_merge_manifest": json.loads(Path(str(PREPARE_MANIFEST["attack_merge_manifest_path"])).read_text(encoding="utf-8")),
        "attack_positive_pool_manifest": json.loads(Path(str(PREPARE_MANIFEST["attack_positive_pool_manifest_path"])).read_text(encoding="utf-8")),
        "attack_negative_pool_manifest": json.loads(Path(str(PREPARE_MANIFEST["attack_negative_pool_manifest_path"])).read_text(encoding="utf-8")),
        "formal_attack_final_decision_metrics": json.loads(Path(str(PREPARE_MANIFEST["formal_attack_final_decision_metrics_path"])).read_text(encoding="utf-8")),
        "formal_attack_attestation_metrics": json.loads(Path(str(PREPARE_MANIFEST["formal_attack_attestation_metrics_path"])).read_text(encoding="utf-8")),
        "derived_attack_union_metrics": json.loads(Path(str(PREPARE_MANIFEST["derived_attack_union_metrics_path"])).read_text(encoding="utf-8")),
        "formal_attack_negative_metrics": json.loads(Path(str(PREPARE_MANIFEST["formal_attack_negative_metrics_path"])).read_text(encoding="utf-8")),
        "quality_pair_plan": json.loads(Path(str(PREPARE_MANIFEST["quality_pair_plan_path"])).read_text(encoding="utf-8")),
        "expected_quality_shard_paths": [
            {"path": str(path_value), "exists": Path(str(path_value)).exists()}
            for path_value in PREPARE_MANIFEST.get("expected_quality_shard_paths", [])
        ],
        "gpu_session_peak_summary": dict(GPU_PEAK_NOTEBOOK_SUMMARY),
    }
elif PW04_MODE == "quality_shard":
    PREPARE_MANIFEST = json.loads(PREPARE_MANIFEST_PATH.read_text(encoding="utf-8"))
    PW04_RESULT_SUMMARY = {
        "mode": PW04_MODE,
        "prepare_manifest": json.loads(PREPARE_MANIFEST_PATH.read_text(encoding="utf-8")),
        "quality_shard": json.loads(SELECTED_QUALITY_SHARD_PATH.read_text(encoding="utf-8")),
        "expected_quality_shard_paths": [
            {"path": str(path_value), "exists": Path(str(path_value)).exists()}
            for path_value in PREPARE_MANIFEST.get("expected_quality_shard_paths", [])
        ],
        "gpu_session_peak_summary": dict(GPU_PEAK_NOTEBOOK_SUMMARY),
    }
else:
    PAPER_SCOPE_REGISTRY_PATH = Path(str(PW04_SUMMARY["paper_scope_registry_path"]))
    CANONICAL_METRICS_PATHS = dict(PW04_SUMMARY.get("canonical_metrics_paths", {}))
    PAPER_TABLE_PATHS = dict(PW04_SUMMARY.get("paper_tables_paths", {}))
    PAPER_FIGURE_PATHS = dict(PW04_SUMMARY.get("paper_figures_paths", {}))
    TAIL_ESTIMATION_PATHS = dict(PW04_SUMMARY.get("tail_estimation_paths", {}))
    BOOTSTRAP_CONFIDENCE_INTERVALS_PATH = Path(str(PW04_SUMMARY["bootstrap_confidence_intervals_path"]))
    BOOTSTRAP_CONFIDENCE_INTERVALS_CSV_PATH = Path(str(PW04_SUMMARY["bootstrap_confidence_intervals_csv_path"]))

    PW04_RESULT_SUMMARY = {
        "mode": PW04_MODE,
        "summary": json.loads(PW04_SUMMARY_PATH.read_text(encoding="utf-8")),
        "formal_attack_final_decision_metrics": json.loads(FORMAL_ATTACK_FINAL_DECISION_METRICS_PATH.read_text(encoding="utf-8")),
        "formal_attack_attestation_metrics": json.loads(FORMAL_ATTACK_ATTESTATION_METRICS_PATH.read_text(encoding="utf-8")),
        "derived_attack_union_metrics": json.loads(DERIVED_ATTACK_UNION_METRICS_PATH.read_text(encoding="utf-8")),
        "clean_attack_overview": json.loads(CLEAN_ATTACK_OVERVIEW_PATH.read_text(encoding="utf-8")),
        "paper_metric_registry": json.loads(PAPER_SCOPE_REGISTRY_PATH.read_text(encoding="utf-8")),
        "content_chain_metrics": json.loads(Path(str(CANONICAL_METRICS_PATHS["content_chain"])).read_text(encoding="utf-8")),
        "event_attestation_metrics": json.loads(Path(str(CANONICAL_METRICS_PATHS["event_attestation"])).read_text(encoding="utf-8")),
        "system_final_metrics": json.loads(Path(str(CANONICAL_METRICS_PATHS["system_final"])).read_text(encoding="utf-8")),
        "bootstrap_confidence_intervals": json.loads(BOOTSTRAP_CONFIDENCE_INTERVALS_PATH.read_text(encoding="utf-8")),
        "bootstrap_confidence_intervals_csv_path": str(BOOTSTRAP_CONFIDENCE_INTERVALS_CSV_PATH),
        "paper_tables_paths": PAPER_TABLE_PATHS,
        "paper_figures_paths": {
            key: {"path": value, "exists": Path(str(value)).exists()}
            for key, value in PAPER_FIGURE_PATHS.items()
        },
        "tail_estimation_paths": TAIL_ESTIMATION_PATHS,
        "estimated_tail_fpr_1e4": json.loads(Path(str(TAIL_ESTIMATION_PATHS["estimated_tail_fpr_1e4_path"])).read_text(encoding="utf-8")),
        "estimated_tail_fpr_1e5": json.loads(Path(str(TAIL_ESTIMATION_PATHS["estimated_tail_fpr_1e5_path"])).read_text(encoding="utf-8")),
        "tail_fit_diagnostics": json.loads(Path(str(TAIL_ESTIMATION_PATHS["tail_fit_diagnostics_path"])).read_text(encoding="utf-8")),
        "tail_fit_stability_summary": json.loads(Path(str(TAIL_ESTIMATION_PATHS["tail_fit_stability_summary_path"])).read_text(encoding="utf-8")),
        "gpu_session_peak_path": GPU_PEAK_NOTEBOOK_SUMMARY.get("gpu_session_peak_path"),
        "gpu_peak_memory_mib": GPU_PEAK_NOTEBOOK_SUMMARY.get("gpu_peak_memory_mib"),
        "gpu_peak_increment_mib": GPU_PEAK_NOTEBOOK_SUMMARY.get("gpu_peak_increment_mib"),
        "peak_gpu_index": GPU_PEAK_NOTEBOOK_SUMMARY.get("peak_gpu_index"),
        "peak_gpu_uuid": GPU_PEAK_NOTEBOOK_SUMMARY.get("peak_gpu_uuid"),
        "peak_gpu_name": GPU_PEAK_NOTEBOOK_SUMMARY.get("peak_gpu_name"),
        "monitor_status": GPU_PEAK_NOTEBOOK_SUMMARY.get("monitor_status"),
        "monitor_recommendation": GPU_PEAK_NOTEBOOK_SUMMARY.get("monitor_recommendation"),
        "monitor_error": GPU_PEAK_NOTEBOOK_SUMMARY.get("monitor_error"),
        "monitor_execution_mode": GPU_PEAK_NOTEBOOK_SUMMARY.get("monitor_execution_mode"),
        "gpu_session_peak_summary": dict(GPU_PEAK_NOTEBOOK_SUMMARY),
    }
print_json("pw04_result_summary", PW04_RESULT_SUMMARY)